# Site4Drug Demo Notebook (Single-Antigen Prediction)

This notebook runs the modality-aware Site4Drug demo with:
- default `mode=auto` modality selection (`epitope` vs `pocket`)
- multi-agent reasoning traces
- overflow-safe token strategy logs
- hydropathy + PTM + candidate-track plot artifacts
        


## 0) Prerequisites

```bash
cd <repo-root>
source .venv/bin/activate
python -m pip install -r requirements.txt
```
        


In [ ]:
from pathlib import Path
import sys
import os


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'README.md').exists() and (candidate / 'pipeline').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.chdir(REPO_ROOT)
print('REPO_ROOT =', REPO_ROOT)
        


In [ ]:
from getpass import getpass

from site4drug_inference.demo.notebook_utils import ensure_api_key_or_raise

try:
    ensure_api_key_or_raise(REPO_ROOT)
    print('TINKER_API_KEY loaded.')
except RuntimeError:
    key = getpass('Paste TINKER_API_KEY (input hidden): ').strip()
    if not key:
        raise RuntimeError('No key provided.')
    os.environ['TINKER_API_KEY'] = key
    ensure_api_key_or_raise(REPO_ROOT)
    print('TINKER_API_KEY set for this notebook session.')
        


## 1) Configure input and mode

- Keep `MODE='auto'` for default modality selection.
- Override `MODE='epitope'` or `MODE='pocket'` if needed.
- Default orchestrator is `react` with bounded retries.
- Default PTM source is `musitedeep`; motif source defaults to `remote`.
- Default checkpoint resolves to `best_epoch1_step400` (from `pipeline/common/model_defaults.py`).


In [ ]:
import importlib
from site4drug_inference.common.model_defaults import BASE_MODEL as _BASE_MODEL
from site4drug_inference.demo import notebook_utils as nb_utils

nb_utils = importlib.reload(nb_utils)
DEFAULT_BASE_MODEL = getattr(nb_utils, "DEFAULT_BASE_MODEL", _BASE_MODEL)
DEFAULT_CHECKPOINT = nb_utils.DEFAULT_CHECKPOINT
resolve_input_sequence = nb_utils.resolve_input_sequence

#UNIPROT = 'P29996'
UNIPROT = 'MUC4'
MODE = 'auto'  # auto | epitope | pocket
CANDIDATE_SOURCE = 'llm_propose'
TOP_K = 5
USE_BASE_MODEL = False  # True -> ignore CHECKPOINT and use BASE_MODEL directly
BASE_MODEL = DEFAULT_BASE_MODEL
CHECKPOINT = DEFAULT_CHECKPOINT
print("DEFAULT_CHECKPOINT =", DEFAULT_CHECKPOINT)
#MAX_TOKENS = 2048
MAX_TOKENS = 10000
MAX_INPUT_TOKENS = 10000
USE_MULTI_AGENT = True
REPAIR_WITH_BASE_MODEL = True
PTM_SOURCE = 'musitedeep'  # musitedeep | hybrid | glyco_only | multi_rule
PTM_POLICY = 'tiered'  # tiered | hard | soft
MOTIF_SOURCE = 'remote'  # local | remote | auto
USE_MOTIF = True

#SEQUENCE_FILE = 'examples/antigens/P29996.fasta'  # set '' to disable
SEQUENCE_FILE = ''
SEQUENCE_TEXT = 'MKGARWRRVPWVSLSCLCLCLLPHVVPGTTEDTLITGSKTAAPVTSTGSTTATLEGQSTAASSRTSNQDISASSQNHQTKSTETTSKAQTDTLTQMMTSTLFSSPSVHNVMETAPPDEMTTSFPSSVTNTLMMTSKTITMTTSTDSTLGNTEETSTAGTESSTPVTSAVSITAGQEGQSRTTSWRTSIQDTSASSQNHWTRSTQTTRESQTSTLTHRTTSTPSFSPSVHNVTGTVSQKTSPSGETATSSLCSVTNTSMMTSEKITVTTSTGSTLGNPGETSSVPVTGSLMPVTSAALVTFDPEGQSPATFSRTSTQDTTAFSKNHQTQSVETTRVSQINTLNTLTPVTTSTVLSSPSGFNPSGTVSQETFPSGETTTSSPSSVSNTFLVTSKVFRMPTSRDSTLGNTEETSLSVSGTISAITSKVSTIWWSDTLSTALSPSSLPPKISTAFHTQQSEGAETTGRPHERSSFSPGVSQEIFTLHETTTWPSSFSSKGHTTWSQTELPSTSTGAATRLVTGNPSTGTAGTIPRVPSKVSAIGEPGEPTTYSSHSTTLPKTTGAGAQTQWTQETGTTGEALLSSPSYSVTQMIKTATSPSSSPMLDRHTSQQITTAPSTNHSTIHSTSTSPQESPAVSQRGHTQAPQTTQESQTTRSVSPMTDTKTVTTPGSSFTASGHSPSEIVPQDAPTISAATTFAPAPTGDGHTTQAPTTALQAAPSSHDATLGPSGGTSLSKTGALTLANSVVSTPGGPEGQWTSASASTSPDTAAAMTHTHQAESTEASGQTQTSEPASSGSRTTSAGTATPSSSGASGTTPSGSEGISTSGETTRFSSNPSRDSHTTQSTTELLSASASHGAIPVSTGMASSIVPGTFHPTLSEASTAGRPTGQSSPTSPSASPQETAAISRMAQTQRTRTSRGSDTISLASQATDTFSTVPPTPPSITSTGLTSPQTETHTLSPSGSGKTFTTALISNATPLPVTYASSASTGHTTPLHVTDASSVSTGHATPLPVTSPSSVSTGHTTPLPVTDTSSESTGHVTPLPVTSFSSASTGDSTPLPVTDTSSASTGHVTPLPVTSLSSASTGDTTPLPVTDTSSASTGHATSLPVTDTSSVSTGHTTPLPVTDTSSASTGHATSLPVTDTSSVSTGHTTPLHVTDASSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATPLLVTDTSSASTGHATPLPVTDASSVSTDHATSLPVTIPSAASTGHTTPLPVTDTSSASTGQATSLLVTDTSSVSTGDTTPLPVTSTSSASTGHVTPLHVTSPSSASTGHATPLPVTSLSSASTGDTMPLPVTSPSSASTGDTTPLPVTDASSVSTGHTTPLHVTDASSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATPLLVTDTSSASTGHATPLPVTDASSVSTDHATSLPVTIPSAASTGHTTPLPVTDTSSASTGQATSLLVTDTSSVSTGDTTPLPVTSTSSASTGHVTPLHVTSPSSASTGHATPLPVTSLSSASTGDTMPLPVTSPSSASTGDTTPLPVTDASSVSTGHTTPLPVTSPSSASTGHTTPLPVTDTSSASKGDTTPLPVTSPSSASTGHTTPLPVTDTSSASTGDTTPLPVTNASSLSTGHATPLHVTSPSSASTGHATPLPVTSTSSASTGHATPLPVTGLSSATTDDTTRLPVTDVSSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHASPLLVTDASSASTGQATPLPVTDTSSVSTAHATPLPVTGLSSASTDDTTRLPVTDVSSASTGQAIPLPVTSPSSASTGDTTPLPVTDASSASTGDTTSLPVTIPSSASSGHTTSLPVTDASSVSTGHATSLLVTDASSVSTGDTTPLPVTDTNSASTGDTTPLHVTDASSVSTGHATSLPVTSLSSASTGDTTPLPVTSPSSASSGHTTPLPVTDASSVPTGHATSLPVTDASSVSTGHATPLPVTDASSVSTGHATPLPVTDTSSVSTGQATPLPVTSLSSASTGDTTPLPVTDTSSASTGQDTPLPVTSLSSVSTGDTTPLPVTNPSSASTGHATPLLVTDASSISTGHATSLLVTDASSVSTGHATALHDTDASSLSTGDTTPLPVTSPSSTSTGDTTPLPVTETSSVSTGHATSLPVTDTSSASTGHATSLPVTDTSSASTGHATPLPVTDTSSASTGQATPLPVTSPSSASTGHAIPLLVTDTSSASTGQATPLPVTSLSSASTGDTTPLPVTDASSVSTGHATSLPVTSLSSVSTGDTTPLPVTSPSSASTGHATPLHVTDASSASTGHATPLPVTSLSSASTGDTTPLPVTSPSSASTGHATPLHVTDASSVSTGDTTPLPVTSSSSASSGHTTPLPVTDASSASTGDTTPLPVTDTSSASTGHATHLPVTGLSSASTGDTTRLPVTNVSSASTGHATPLPVTSTSSASTGDTTPLPGTDTSSVSTGHTTPLLVTDASSVSTGDTTRLPVTSPSSASTGHTTPLPVTDTPSASTGDTTPLPVTNASSLSTRHATSLHVTSPSSASTGHATSLPVTDTSAASTGHATPLPVTSTSSASTGDTTPLPVTDTYSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATPLLVTDASSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATSLPVTDTSSASTGDTTSLPVTDTSSAYTGDTTSLPVTDTSSSSTGDTTPLLVTETSSVSTGDTTPLPVTDTSSASTGHATPLPVTNTSSVSTGHATPLHVTSPSSASTGHTTPLPVTDASSVSTGHATSLPVTDASSVFTGHATSLPVTIPSSASSGHTTPLPVTDASSVSTGHATSLPVTDASSVSTGHATPLPVTDASSVSTGHATPLPLTSLSSVSTGDTTPLPVTDTSSASTGQATPLPVTSLSSVSTGDTTPLPVTDTSSASTGHATSLPVTDTSSASTGHATPLPDTDTSSASTGHATLLPVTDTSSASIGHATSLPVTDTSSISTGHATPLHVTSPSSASTGHATPLPVTDTSSASTGHANPLHVTSPSSASTGHATPLPVTDTSSASTGHATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHTTPLPVTDTSSASTGQATALPVTSTSSASTGDTTPLPVTDTSSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATPLLVTDASSASTGQATPLPVTSLSSVSTGDTTPLPVTSPSSASTGHATSLPVTDTSSASTGDTTSLPVTDTSSAYTGDTTSLPVTDTSSSSTGDTTPLLVTETSSVSTGHATPLLVTDASSASTGHATPLHVTSPSSASTGDTTPVPVTDTSSVSTGHATPLPVTGLSSASTGDTTRLPVTDISSASTGQATPLPVTNTSSVSTGDTMPLPVTSPSSASTGHATPLPVTSTSSASTGHATPVPVTSTSSASTGHTTPLPVTDTSSASTGDTTPLPVTSPSSASTGHTTPLHVTIPSSASTGDTSTLPVTGASSASTGHATPLPVTDTSSVSTGHATPLPVTSLSSVSTGDTTPLPVTDASSASTGQATPLPVTSLSSVSTGDTTPLLVTDASSVSTGHATPLPVTDTSSASTGDTTRLPVTDTSSASTGQATPLPVTSLSSVSTGDTTPLLVTDASSVSTGHATPLPVTDTSSASTGDTTRLPVTDTSSASTGQATPLPVTIPSSSSSGHTTPLPVTSTSSVSTGHVTPLHVTSPSSASTGHVTPLPVTSTSSASTGHATPLLVTDASSVSTGHATPLPVTDASSASTGDTTPLPVTDTSSASTGQATPLPVTSLSSVSTGDTTPLPVTDASSASTGHATPLPVTIPSSVSTGDTMPLPVTSPSSASTGHATPLPVTGLSSASTGDTTPLPVTDTSSASTRHATPLPVTDTSSASTDDTTRLPVTDVSSASTGHATPLPVTSTSSASTGDTTPLPVTDTSSVSTGHATSLPVTSRSSASTGHATPLPVTDTSSVSTGHATPLPVTSTSSVSTGHATPLPVTSPSSASTGHATPVPVTSTSSASTGDTTPLPVTNASSLSTGHATPLHVTSPSSASRGDTSTLPVTDASSASTGHATPLPLTSLSSVSTGDTTPLPVTDTSSASTGQATPLPVTSLSSVSTGDTTPLPVTIPSSASSGHTTSLPVTDASSVSTGHGTPLPVTSTSSASTGDTTPLPVTDTSSASTGHATPLPVTDTSSASTGHATPLPVTSLSSVSTGHATPLAVSSATSASTVSSDSPLKMETPGMTTPSLKTDGGRRTATSPPPTTSQTIISTIPSTAMHTRSTAAPIPILPERGVSLFPYGAGAGDLEFVRRTVDFTSPLFKPATGFPLGSSLRDSLYFTDNGQIIFPESDYQIFSYPNPLPTGFTGRDPVALVAPFWDDADFSTGRGTTFYQEYETFYGEHSLLVQQAESWIRKMTNNGGYKARWALKVTWVNAHAYPAQWTLGSNTYQAILSTDGSRSYALFLYQSGGMQWDVAQRSGNPVLMGFSSGDGYFENSPLMSQPVWERYRPDRFLNSNSGLQGLQFYRLHREERPNYRLECLQWLKSQPRWPSWGWNQVSCPCSWQQGRRDLRFQPVSIGRWGLGSRQLCSFTSWRGGVCCSYGPWGEFREGWHVQRPWQLAQELEPQSWCCRWNDKPYLCALYQQRRPHVGCATYRPPQPAWMFGDPHITTLDGVSYTFNGLGDFLLVGAQDGNSSFLLQGRTAQTGSAQATNFIAFAAQYRSSSLGPVTVQWLLEPHDAIRVLLDNQTVTFQPDHEDGGGQETFNATGVLLSRNGSEVSASFDGWATVSVIALSNILHASASLPPEYQNRTEGLLGVWNNNPEDDFRMPNGSTIPPGSPEEMLFHFGMTWQINGTGLLGKRNDQLPSNFTPVFYSQLQKNSSWAEHLISNCDGDSSCIYDTLALRNASIGLHTREVSKNYEQANATLNQYPPSINGGRVIEAYKGQTTLIQYTSNAEDANFTLRDSCTDLELFENGTLLWTPKSLEPFTLEILARSAKIGLASALQPRTVVCHCNAESQCLYNQTSRVGNSSLEVAGCKCDGGTFGRYCEGSEDACEEPCFPSVHCVPGKGCEACPPNLTGDGRHCAALGSSFLCQNQSCPVNYCYNQGHCYISQTLGCQPMCTCPPAFTDSRCFLAGNNFSPTVNLELPLRVIQLLLSEEENASMAEVNASVAYRLGTLDMRAFLRNSQVERIDSAAPASGSPIQHWMVISEFQYRPRGPVIDFLNNQLLAAVVEAFLYHVPRRSEEPRNDVVFQPISGEDVRDVTALNVSTLKAYFRCDGYKGYDLVYSPQSGFTCVSPCSRGYCDHGGQCQHLPSGPRCSCVSFSIYTAWGEHCEHLSMKLDAFFGIFFGALGGLLLLGVGTFVVLRFWGCSGARFSYFLNSAEALP'  # paste sequence here if preferred

raw_sequence, input_source = resolve_input_sequence(
    uniprot=UNIPROT,
    sequence_text=SEQUENCE_TEXT,
    sequence_file=SEQUENCE_FILE,
    allow_online_lookup=True,
)
print('Input source:', input_source)
print('Sequence length:', len(raw_sequence))


## 2) Run prediction


In [ ]:
from site4drug_inference.demo.notebook_utils import run_notebook_prediction

result = run_notebook_prediction(
    uniprot=UNIPROT,
    raw_sequence=raw_sequence,
    checkpoint=CHECKPOINT,
    base_model=BASE_MODEL,
    use_base_model=USE_BASE_MODEL,
    mode=MODE,
    candidate_source=CANDIDATE_SOURCE,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_input_tokens=MAX_INPUT_TOKENS,
    output_dir=REPO_ROOT / 'outputs' / 'predictions',
    enable_plot=True,
    use_multi_agent=USE_MULTI_AGENT,
    input_source=input_source,
    repair_with_base_model=REPAIR_WITH_BASE_MODEL,
    ptm_source=PTM_SOURCE,
    ptm_policy=PTM_POLICY,
    motif_source=MOTIF_SOURCE,
    use_motif=USE_MOTIF,
)

run_payload = result['run_payload']
run_dir = result['run_dir']
print('Run dir:', run_dir)
print('Recommended modality:', run_payload['recommended_modality'])
print('Modality confidence:', run_payload['modality_confidence'])
print('Token strategy:', run_payload['token_strategy_used'])
print('PTM summary:', run_payload.get('ptm_summary', {}))
print('Motif summary:', run_payload.get('motif_summary', {}))
print('Orchestrator steps:', len(run_payload.get('orchestrator_trace', [])))
print('Parsed count:', run_payload['parsed_candidate_count'])


## 3) Inspect ranked candidates, evidence, and plot


In [ ]:
import pandas as pd
from IPython.display import HTML, Image, display

from site4drug_inference.demo.notebook_utils import ranked_candidates_df, candidate_evidence_df

rank_df = ranked_candidates_df(run_payload)
ev_df = candidate_evidence_df(run_payload)

display(rank_df)
display(ev_df)

plot_name = run_payload.get('plot_artifacts', {}).get('plot_png_name')
if plot_name:
    display(Image(filename=str(run_dir / plot_name)))
else:
    print('No plot artifact generated.')

display(HTML((run_dir / 'prediction_report.html').read_text(encoding='utf-8')))
        


## 4) Raw output, token strategy, PTM/motif summaries, and traces


In [ ]:
from pprint import pprint

print('Token budget events:')
pprint(run_payload.get('token_budget_events', []))

print('\nPTM summary:')
pprint(run_payload.get('ptm_summary', {}))

print('\nMotif summary:')
pprint(run_payload.get('motif_summary', {}))

print('\nOrchestrator trace (first 10 rows):')
pprint(run_payload.get('orchestrator_trace', [])[:10])

print('\nAudit log:')
pprint(run_payload.get('audit_log', {}))

print('\nAgent trace keys:', list(run_payload.get('agent_traces', {}).keys()))
print('\nRaw model output preview:')
print(run_payload.get('raw_model_output', '')[:2000])
